# LumenY 11 — Evaluation & Experimentation

Loads `model_4H_long.joblib` and `model_4H_short.joblib` from `models_11/`.  
No retraining. Evaluates signal quality, PnL by threshold, pair, hour, month.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
import gc
from pathlib import Path
from sklearn.metrics import roc_auc_score, log_loss
warnings.filterwarnings('ignore')

DATA_DIR   = Path('../backend/data/features_combined')
PRICE_DIR  = Path('../backend/data/processed')
MODELS_DIR = Path('../backend/models_11')

MAJORS    = ['EURUSD', 'GBPUSD', 'USDJPY', 'USDCHF', 'USDCAD', 'AUDUSD', 'NZDUSD']
TRAIN_END = '2024-06-30'
ATR_MULT  = 0.75

SPREADS = {
    'EURUSD': 0.2 * 0.0001,
    'GBPUSD': 0.4 * 0.0001,
    'USDJPY': 0.3 * 0.01 / 150,
    'USDCHF': 0.4 * 0.0001,
    'USDCAD': 0.4 * 0.0001,
    'AUDUSD': 0.3 * 0.0001,
    'NZDUSD': 0.5 * 0.0001,
}

COLS_DROP_SUFFIX = '_f9'
COLS_DROP_SUBSTR = '_1W'
LOOKAHEAD_COLS   = {
    'mfe_long_pips', 'mfe_short_pips',
    'trail_long_bars', 'trail_short_bars',
    'trail_stop_pips', 'mfe_atr_24',
}

print('Ready.')
print('Spreads (log return):')
for p, s in SPREADS.items():
    print(f'  {p}: {s:.6f}')

## 1. Load Test Data + Build Labels

In [ ]:
bundle       = joblib.load(MODELS_DIR / 'model_4H_long.joblib')
feature_cols = bundle['feature_cols']
del bundle

dfs_test = []
for pair in MAJORS:
    print(f'  {pair}...', flush=True)
    df = pd.read_parquet(DATA_DIR / f'{pair}_combined.parquet')
    drop = [c for c in df.columns if (
        c.endswith(COLS_DROP_SUFFIX) or COLS_DROP_SUBSTR in c or
        c == 'pair_id' or c in LOOKAHEAD_COLS
    )]
    df.drop(columns=drop, errors='ignore', inplace=True)

    price_df = pd.read_parquet(PRICE_DIR / f'{pair}_1H.parquet')
    close = price_df['close']
    high  = price_df['high']
    low   = price_df['low']

    tr  = pd.concat([high - low,
                     (high - close.shift()).abs(),
                     (low  - close.shift()).abs()], axis=1).max(axis=1)
    atr = tr.rolling(14).mean()

    close_arr = close.reindex(df.index).values
    high_arr  = high.reindex(df.index).values
    low_arr   = low.reindex(df.index).values
    atr_arr   = atr.reindex(df.index).values
    n = len(df)

    label_long  = np.full(n, np.nan)
    label_short = np.full(n, np.nan)
    for i in range(n - 4):
        if np.isnan(atr_arr[i]) or np.isnan(close_arr[i]):
            continue
        thresh       = ATR_MULT * atr_arr[i]
        long_target  = close_arr[i] + thresh
        short_target = close_arr[i] - thresh
        long_hit = short_hit = 0
        for h in range(1, 5):
            j = i + h
            if high_arr[j] >= long_target:  long_hit  = 1
            if low_arr[j]  <= short_target: short_hit = 1
        label_long[i]  = long_hit
        label_short[i] = short_hit

    df['label_long']  = label_long
    df['label_short'] = label_short
    df['pair'] = pair

    feat_cols_present = [c for c in df.columns if c not in ('label_long', 'label_short', 'pair')]
    df[feat_cols_present] = df[feat_cols_present].astype(np.float32)

    # also store log return for PnL calc
    df['log_ret_4H'] = np.log(close.shift(-4) / close).reindex(df.index)

    dfs_test.append(df[df.index > TRAIN_END])
    del df; gc.collect()

df_test = pd.concat(dfs_test).sort_index(); del dfs_test; gc.collect()

valid = df_test['label_long'].notna()
X_test        = df_test.loc[valid, feature_cols].ffill().fillna(0)
y_long_test   = df_test.loc[valid, 'label_long'].astype(np.int8)
y_short_test  = df_test.loc[valid, 'label_short'].astype(np.int8)
log_ret_test  = df_test.loc[valid, 'log_ret_4H'].astype(np.float32)
pair_test     = df_test.loc[valid, 'pair']
del df_test; gc.collect()

print(f'X_test: {X_test.shape}  ({X_test.index.min().date()} -> {X_test.index.max().date()})')
print(f'Long hit rate:  {y_long_test.mean():.1%}')
print(f'Short hit rate: {y_short_test.mean():.1%}')

## 2. Predict Probabilities

In [ ]:
bundle_long  = joblib.load(MODELS_DIR / 'model_4H_long.joblib')
bundle_short = joblib.load(MODELS_DIR / 'model_4H_short.joblib')

p_long  = bundle_long['model'].predict_proba(X_test)[:, 1]
p_short = bundle_short['model'].predict_proba(X_test)[:, 1]
del bundle_long, bundle_short; gc.collect()

auc_long  = roc_auc_score(y_long_test,  p_long)
auc_short = roc_auc_score(y_short_test, p_short)
print(f'Test AUC — long:  {auc_long:.4f}')
print(f'Test AUC — short: {auc_short:.4f}')

results = pd.DataFrame({
    'p_long':    p_long,
    'p_short':   p_short,
    'hit_long':  y_long_test.values,
    'hit_short': y_short_test.values,
    'log_ret':   log_ret_test.values,
    'pair':      pair_test.values,
}, index=X_test.index)
results['spread'] = results['pair'].map(SPREADS)
results['hour']   = results.index.hour
del X_test; gc.collect()

print(f'\nResults: {len(results):,} rows')

## 3. OOF Calibration Check

In [ ]:
# OOF probs stored in training notebook memory — reload from npy
# (only available if training notebook was run in same session)
import os
oof_long_path  = MODELS_DIR / 'oof_probs_long.npy'
oof_short_path = MODELS_DIR / 'oof_probs_short.npy'
y_long_path    = MODELS_DIR / 'y_long_train.npy'
y_short_path   = MODELS_DIR / 'y_short_train.npy'

# Save OOF probs from training notebook (run this after training if available)
# np.save(oof_long_path, oof_probs_long)
# np.save(oof_short_path, oof_probs_short)

if oof_long_path.exists() and y_long_path.exists():
    for name, oof_path, y_path in [
        ('long',  oof_long_path,  y_long_path),
        ('short', oof_short_path, y_short_path),
    ]:
        oof  = np.load(oof_path)
        y    = np.load(y_path)
        valid = ~np.isnan(oof)
        probs  = oof[valid]
        y_true = y[valid].astype(int)
        auc = roc_auc_score(y_true, probs)
        ll  = log_loss(y_true, probs)
        base_ll = log_loss(y_true, np.full(len(y_true), y_true.mean()))
        print(f'{name.upper()} OOF: AUC={auc:.4f}  LogLoss={ll:.4f}  Baseline={base_ll:.4f}')
        print(f'  {"Prob range":<12} {"N":>8} {"Hit rate":>9} {"Edge":>8}')
        print('  ' + '-'*42)
        for lo, hi in [(0.0,0.3),(0.3,0.4),(0.4,0.5),(0.5,0.6),(0.6,0.7),(0.7,1.0)]:
            mask = (probs >= lo) & (probs < hi)
            if mask.sum() < 50: continue
            hit  = y_true[mask].mean()
            edge = hit - y_true.mean()
            print(f'  [{lo:.1f}-{hi:.1f})  {mask.sum():>8,}   {hit:>8.1%}   {edge:>+7.1%}')
        print()
else:
    print('OOF files not found. To generate:')
    print('  1. Run training notebook')
    print('  2. Add: np.save(MODELS_DIR / "oof_probs_long.npy", oof_probs_long)')
    print('          np.save(MODELS_DIR / "oof_probs_short.npy", oof_probs_short)')
    print('  3. Re-run this cell')

## 4. Threshold Sweep — Signal Quality

In [ ]:
# Signal rule: long when p_long > t_L AND p_short < (1-t_S)
#              short when p_short > t_S AND p_long < (1-t_L)
# Sweep symmetric thresholds

nm = (results.index.max() - results.index.min()).days / 30

def eval_direction(sub, direction):
    if len(sub) < 30: return None
    if direction == 'long':
        pnl = sub['log_ret'] - sub['spread']
        hit = sub['hit_long']
    else:
        pnl = -sub['log_ret'] - sub['spread']
        hit = sub['hit_short']
    ev  = pnl.mean()
    wr  = (pnl > 0).mean()
    sh  = (ev / pnl.std()) * np.sqrt(252 * 24) if pnl.std() > 0 else 0
    pmo = pnl.sum() / nm
    atr_hit = hit.mean()
    return {'n': len(sub), 'wr': wr, 'ev': ev, 'sh': sh, 'pmo': pmo, 'atr_hit': atr_hit}

rows = []
for t in [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    for direction, p_col, opp_col in [('long', 'p_long', 'p_short'), ('short', 'p_short', 'p_long')]:
        mask = results[p_col] > t
        sub  = results[mask]
        s = eval_direction(sub, direction)
        if s: rows.append({'thresh': t, 'dir': direction, **s})

    # combined (only take trade when clear directional disagreement)
    long_mask  = (results['p_long'] > t) & (results['p_short'] < (1 - t))
    short_mask = (results['p_short'] > t) & (results['p_long'] < (1 - t))
    long_sub   = results[long_mask]
    short_sub  = results[short_mask]
    sl = eval_direction(long_sub,  'long')
    ss = eval_direction(short_sub, 'short')
    if sl and ss:
        n_tot = sl['n'] + ss['n']
        rows.append({
            'thresh': t, 'dir': 'combined',
            'n': n_tot,
            'wr': (sl['wr']*sl['n'] + ss['wr']*ss['n']) / n_tot,
            'ev': (sl['ev']*sl['n'] + ss['ev']*ss['n']) / n_tot,
            'sh': (sl['sh'] + ss['sh']) / 2,
            'pmo': sl['pmo'] + ss['pmo'],
            'atr_hit': (sl['atr_hit']*sl['n'] + ss['atr_hit']*ss['n']) / n_tot,
        })

df_thresh = pd.DataFrame(rows)
print(f'{"Thresh":>7} {"Dir":<10} {"N":>7} {"WR":>7} {"EV":>10} {"Sharpe":>8} {"PnL/mo":>10} {"ATR%":>7}')
print('=' * 75)
for _, r in df_thresh.iterrows():
    flag = ' <<<' if r['ev'] > 0 else ''
    print(f'{r.thresh:>7.2f} {r.dir:<10} {r.n:>7,} {r.wr:>7.1%} {r.ev:>+10.6f} {r.sh:>+8.2f} {r.pmo:>+10.4f} {r.atr_hit:>7.1%}{flag}')

## 5. Best Config Deep Dive

In [ ]:
# Pick best combined config by PnL/mo
best_row = df_thresh[df_thresh['dir'] == 'combined'].sort_values('pmo', ascending=False).iloc[0]
BEST_THRESH = best_row['thresh']
print(f'Best threshold: {BEST_THRESH:.2f}  PnL/mo={best_row.pmo:+.4f}  Sharpe={best_row.sh:+.2f}  N={best_row.n:,}')

long_mask  = (results['p_long']  > BEST_THRESH) & (results['p_short'] < (1 - BEST_THRESH))
short_mask = (results['p_short'] > BEST_THRESH) & (results['p_long']  < (1 - BEST_THRESH))

sub_long  = results[long_mask].copy()
sub_short = results[short_mask].copy()
sub_long['pnl']  =  sub_long['log_ret']  - sub_long['spread']
sub_short['pnl'] = -sub_short['log_ret'] - sub_short['spread']
sub_long['direction']  = 'long'
sub_short['direction'] = 'short'
sub = pd.concat([sub_long, sub_short]).sort_index()

print(f'\n  Long trades:  {len(sub_long):,}   Short trades: {len(sub_short):,}')
print(f'  Overall WR:   {(sub["pnl"] > 0).mean():.1%}')
print(f'  Overall EV:   {sub["pnl"].mean():+.6f}')

# Per-pair
print(f'\nPER-PAIR:')
print(f'  {"Pair":<10} {"N":>6} {"WR":>7} {"EV":>10} {"Sharpe":>8} {"PnL/mo":>10}')
print('  ' + '-'*55)
for pair in sorted(sub['pair'].unique()):
    p   = sub[sub['pair'] == pair]
    wr  = (p['pnl'] > 0).mean()
    ev  = p['pnl'].mean()
    sh  = (ev / p['pnl'].std()) * np.sqrt(252 * 24) if p['pnl'].std() > 0 else 0
    pmo = p['pnl'].sum() / nm
    flag = ' <<<' if ev > 0 else ''
    print(f'  {pair:<10} {len(p):>6,} {wr:>7.1%} {ev:>+10.6f} {sh:>+8.2f} {pmo:>+10.4f}{flag}')

# Hour of day
print(f'\nHOUR OF DAY (UTC):')
print(f'  {"Hour":>5} {"N":>6} {"WR":>7} {"EV":>10} {"PnL/mo":>10}')
print('  ' + '-'*45)
for h, g in sub.groupby('hour'):
    if len(g) < 20: continue
    wr  = (g['pnl'] > 0).mean()
    ev  = g['pnl'].mean()
    pmo = g['pnl'].sum() / nm
    flag = ' <<<' if ev > 0 else ''
    print(f'  {h:>5}H  {len(g):>6,} {wr:>7.1%} {ev:>+10.6f} {pmo:>+10.4f}{flag}')

# Monthly
print(f'\nMONTHLY:')
print(f'  {"Month":<10} {"N":>5} {"WR":>7} {"EV":>10} {"CumPnL":>10}')
print('  ' + '-'*48)
cum = 0
for (yr, mo), g in sub.groupby([sub.index.year, sub.index.month]):
    ev  = g['pnl'].mean()
    cum += g['pnl'].sum()
    wr  = (g['pnl'] > 0).mean()
    flag = ' <--' if ev < 0 else ''
    print(f'  {yr}-{mo:02d}    {len(g):>5} {wr:>7.1%} {ev:>+10.6f} {cum:>+10.4f}{flag}')

# PnL distribution
print(f'\nPnL DISTRIBUTION:')
pnl    = sub['pnl']
wins   = pnl[pnl > 0]
losses = pnl[pnl <= 0]
print(f'  Avg win:  {wins.mean():+.6f}   Avg loss: {losses.mean():+.6f}   Ratio: {abs(wins.mean()/losses.mean()):.2f}x')
print(f'  Max win:  {pnl.max():+.6f}   Max loss: {pnl.min():+.6f}')
print(f'  Std:      {pnl.std():.6f}')
buckets = [(-1,-0.003),(-0.003,-0.001),(-0.001,0),(0,0.001),(0.001,0.003),(0.003,1)]
for lo, hi in buckets:
    n   = ((pnl > lo) & (pnl <= hi)).sum()
    pct = n / len(pnl)
    bar = '#' * int(pct * 40)
    print(f'  [{lo:+.3f} to {hi:+.3f}]: {n:>6,} ({pct:>5.1%}) {bar}')

## 6. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(20, 10))
fig.patch.set_facecolor('#080c14')

for ax, name in zip(axes, ['long', 'short']):
    bundle     = joblib.load(MODELS_DIR / f'model_4H_{name}.joblib')
    importance = pd.Series(bundle['model'].feature_importances_, index=feature_cols)
    importance = importance.sort_values(ascending=True).tail(30)
    ax.barh(importance.index, importance.values, color='#4fc3f7', alpha=0.8)
    ax.set_facecolor('#080c14')
    ax.tick_params(colors='white', labelsize=7)
    ax.set_title(f'{name.upper()} model — Top 30 Features', color='white')
    for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')
    del bundle

plt.suptitle('Feature Importance — Dual Binary 4H Model', color='white', fontsize=13)
plt.tight_layout()
plt.show()